# Custom Mojo Ops 🚀

> Injecting pure Mojo kernels into the MAX Graph.

In [ ]:
#| default_exp custom_ops

In [ ]:
#| export
from typing import Any, Dict, List, Union
import pyarrow as pa
import pyarrow.compute as pc
import numpy as np
import time as _time
from pathlib import Path
from math import ceil
from max import engine, driver
from max.graph import Graph, TensorType, ops
from max.graph.type import DType, DeviceRef
from mxframe.lazy_expr import Expr
from mxframe.lazy_frame import LogicalPlan, Scan, Filter, Project, Aggregate, Sort, Limit, Distinct, Join
from mxframe.compiler import GraphCompiler, _max_dtype


## The `CustomOpsCompiler` Class 🛠️

This class extends `GraphCompiler` to support custom Mojo kernels.

In [ ]:
#| export
KERNELS_PATH = (
    str(Path(__file__).parent / "kernels_v261")
    if "__file__" in dir()
    else str(Path("/home/ablearn/mxdf/mxframe/kernels_v261"))
)

WARP_SIZE = 32
MAX_GROUPS = 64
AUTO_GPU_THRESHOLD = 100_000   # rows; used by device='auto' to pick GPU vs CPU

# ── Module-level caches ──────────────────────────────────────────────────────
# Persist across CustomOpsCompiler instances so repeated .compute() calls
# reuse the compiled MAX Graph model instead of recompiling from scratch.
_SESSION_CACHE: Dict[str, Any] = {}       # device_name → InferenceSession
_MODEL_CACHE: Dict[tuple, tuple] = {}     # cache_key → (model, agg_names)
_GROUP_ENCODE_CACHE: Dict[tuple, tuple] = {}  # (table_id, num_rows, keys) → encoding result
_INPUT_PREP_CACHE: Dict[tuple, dict] = {}    # (table_id, num_rows, plan_sig, device) → prepared arrays
_POST_OP_MODEL_CACHE: Dict[tuple, Any] = {}  # ("sort_indices"|"unique_mask", N, device) → model  (GPU only)


def _get_or_create_session(device_name: str, driver_device) -> Any:
    """Return a cached InferenceSession, creating one if needed."""
    if device_name not in _SESSION_CACHE:
        _SESSION_CACHE[device_name] = engine.InferenceSession(devices=[driver_device])
    return _SESSION_CACHE[device_name]


def clear_cache():
    """Clear the compiled model cache and session cache.

    Call this after modifying Mojo kernel source files, or to force
    recompilation on the next .compute() call.
    """
    _MODEL_CACHE.clear()
    _SESSION_CACHE.clear()
    _GROUP_ENCODE_CACHE.clear()
    _INPUT_PREP_CACHE.clear()
    _POST_OP_MODEL_CACHE.clear()


class CustomOpsCompiler(GraphCompiler):
    """Compiles a LogicalPlan into a MAX Graph with custom Mojo kernels.

    Extends GraphCompiler with:
    1. Pre-filter   -- inherited _strip_filters applies Filter nodes in PyArrow.
    2. Group ids    -- PyArrow dictionary-encodes group-by keys into int32 ids.
    3. Kernel dispatch -- grouped sum/min/max/mean/count routed to Mojo kernels.
    4. Keys in result -- group-by key columns are prepended to the output table.
    5. GPU path     -- kernels run on GPU when device='gpu' or device='auto' (N > 100K).
    6. Graph caching -- compiled models are cached by plan signature + input schema.
    7. Post-ops     -- Sort/Limit/Distinct applied via Mojo kernels after graph execution.
    """

    def __init__(self, kernels_path: str = None, device: str = "cpu"):
        super().__init__(device=device)
        self.kernels_path = kernels_path or KERNELS_PATH
        self.last_compile_provenance = {}
        # Reuse module-level cached session to avoid InferenceSession overhead
        self.session = _get_or_create_session(self._session_device, self._driver_device)

    def _maybe_switch_device(self, N: int) -> None:
        """For device='auto': re-evaluate CPU vs GPU based on row count.

        Switches and re-creates the InferenceSession only when the decision changes.
        Threshold: N > AUTO_GPU_THRESHOLD and GPU available -> GPU, otherwise CPU.
        """
        if self._device_arg != "auto":
            return
        want_gpu = self._has_gpu and N > AUTO_GPU_THRESHOLD
        have_gpu = (self._session_device == "gpu")
        if want_gpu == have_gpu:
            return
        if want_gpu:
            self._device_ref = DeviceRef.GPU(0)
            self.session = _get_or_create_session("gpu", self._gpu_driver)
            self._session_device = "gpu"
        else:
            self._device_ref = DeviceRef.CPU()
            self.session = _get_or_create_session("cpu", driver.CPU())
            self._session_device = "cpu"

    @staticmethod
    def _normalize_numeric_dtype(arr: np.ndarray) -> Union[np.ndarray, None]:
        """Normalize numeric arrays to MAX-friendly dtypes."""
        kind = arr.dtype.kind
        if kind in ("i", "u"):
            if arr.dtype not in (np.dtype("int32"), np.dtype("int64")):
                return arr.astype(np.int64, copy=False)
            return arr
        if kind == "f":
            if arr.dtype not in (np.dtype("float32"), np.dtype("float64")):
                return arr.astype(np.float64, copy=False)
            return arr
        if kind == "b":
            return arr.astype(np.bool_, copy=False)
        return None

    @staticmethod
    def _arrow_array_to_numpy_view(arr: pa.Array) -> Union[np.ndarray, None]:
        """Try zero-copy NumPy view from Arrow primitive array buffers."""
        if arr.null_count != 0:
            return None
        typ = arr.type
        if pa.types.is_int32(typ):
            dtype = np.int32
            itemsize = 4
        elif pa.types.is_int64(typ):
            dtype = np.int64
            itemsize = 8
        elif pa.types.is_uint32(typ):
            dtype = np.uint32
            itemsize = 4
        elif pa.types.is_uint64(typ):
            dtype = np.uint64
            itemsize = 8
        elif pa.types.is_float32(typ):
            dtype = np.float32
            itemsize = 4
        elif pa.types.is_float64(typ):
            dtype = np.float64
            itemsize = 8
        else:
            return None

        buffers = arr.buffers()
        if len(buffers) < 2 or buffers[1] is None:
            return None

        offset_bytes = arr.offset * itemsize
        try:
            return np.frombuffer(buffers[1], dtype=dtype, count=len(arr), offset=offset_bytes)
        except Exception:
            return None

    @staticmethod
    def _is_numeric_arrow_type(typ) -> bool:
        """Return True for Arrow types we can convert to numpy tensors."""
        return (pa.types.is_integer(typ) or pa.types.is_floating(typ)
                or pa.types.is_boolean(typ))

    def _column_to_numpy_cached(self, col: Union[pa.Array, pa.ChunkedArray]) -> tuple[Union[np.ndarray, None], bool, bool]:
        """Convert Arrow column to numeric NumPy with zero-copy-first strategy."""
        if isinstance(col, pa.ChunkedArray):
            if len(col.chunks) == 1:
                arr = col.chunks[0]
            else:
                arr = col.combine_chunks()
        else:
            arr = col

        view = self._arrow_array_to_numpy_view(arr)
        if view is not None:
            norm = self._normalize_numeric_dtype(view)
            if norm is not None:
                out = norm if norm.flags['C_CONTIGUOUS'] else np.ascontiguousarray(norm)
                return out, False, True

        try:
            np_arr = arr.to_numpy(zero_copy_only=True)
            norm = self._normalize_numeric_dtype(np_arr)
            if norm is not None:
                out = norm if norm.flags['C_CONTIGUOUS'] else np.ascontiguousarray(norm)
                return out, False, False
        except Exception:
            pass

        try:
            np_arr = arr.to_numpy(zero_copy_only=False)
            norm = self._normalize_numeric_dtype(np_arr)
            if norm is not None:
                out = norm if norm.flags['C_CONTIGUOUS'] else np.ascontiguousarray(norm)
                return out, True, False
        except Exception:
            return None, False, False
        return None, False, False

    def _to_gpu_input(self, arr: np.ndarray) -> tuple[Any, str]:
        """Build GPU input buffer using fastest available safe path."""
        try:
            return driver.Buffer.from_numpy(arr).to(self._gpu_driver), "from_numpy"
        except Exception:
            contiguous = np.ascontiguousarray(arr)
            return driver.Buffer.from_numpy(contiguous).to(self._gpu_driver), "from_numpy_contiguous"

    def _compute_cache_key(self, plan, all_names, all_arrays, n_groups):
        """Compute a hashable cache key for the compiled graph model.

        Key components:
        - plan_sig: structural signature of the logical plan (ops, column refs, literals)
        - input_sig: (name, dtype, shape) per input tensor -- determines graph input types
        - n_groups: number of groups (affects output shapes in grouped path)
        - device: CPU vs GPU (different kernel dispatch + output shapes)
        """
        plan_sig = plan.signature()
        input_sig = tuple(
            (n, str(all_arrays[n].dtype), all_arrays[n].shape[0]) for n in all_names
        )
        return (plan_sig, input_sig, n_groups, self._session_device)

    @classmethod
    def _collect_predicates(cls, plan):
        """Collect all Filter predicates from the plan tree."""
        preds = []
        def _walk(p):
            if isinstance(p, Filter):
                preds.append(p.predicate)
                _walk(p.input)
            elif isinstance(p, Join):
                raise RuntimeError("_collect_predicates: Join should have been materialized")
            elif isinstance(p, (Sort, Limit, Distinct)):
                _walk(p.input)
            elif hasattr(p, 'input'):
                _walk(p.input)
        _walk(plan)
        return preds

    @classmethod
    def _strip_filter_nodes(cls, plan):
        """Remove Filter nodes from the plan tree without applying them."""
        if isinstance(plan, Scan): return plan
        if isinstance(plan, Join):
            raise RuntimeError("_strip_filter_nodes: Join should have been materialized")
        if isinstance(plan, Filter):
            return cls._strip_filter_nodes(plan.input)
        if isinstance(plan, Project):
            return Project(cls._strip_filter_nodes(plan.input), plan.exprs)
        if isinstance(plan, Aggregate):
            return Aggregate(cls._strip_filter_nodes(plan.input), plan.group_by, plan.aggs)
        if isinstance(plan, Sort):
            return Sort(cls._strip_filter_nodes(plan.input), plan.by, plan.descending)
        if isinstance(plan, Limit):
            return Limit(cls._strip_filter_nodes(plan.input), plan.n)
        if isinstance(plan, Distinct):
            return Distinct(cls._strip_filter_nodes(plan.input), plan.subset)
        if hasattr(plan, 'input'):
            plan.input = cls._strip_filter_nodes(plan.input)
        return plan

    def compile_and_run(self, plan, verbose=False) -> pa.Table:
        """Optimized execution with three-tier caching + post-ops.

        Pipeline:
        -1. Materialize any Join nodes (recursive, bottom-up).
        0. Extract post-ops (Sort/Limit/Distinct) from top of plan.
        1. Check input-prep cache; on hit skip to step 5.
        2. Pre-encode group keys on original table (cached).
        3. Compute filter mask as numpy boolean array.
        4. Apply mask via numpy boolean indexing; store in prep cache.
        5. Look up / compile cached MAX Graph model.
        6. Execute model and assemble output.
        7. Apply post-ops via Mojo kernels.
        """
        t0 = _time.perf_counter()

        # -1 -- Materialize Join nodes (bottom-up) --------------------
        plan = self._materialize_joins(plan)

        # 0 -- Extract post-ops (Sort/Limit/Distinct) -----------------
        plan, post_ops = self._extract_post_ops(plan)

        # 0b -- No-compute shortcut: when plan has no Aggregate/Project,
        # process in PyArrow (preserving all column types including strings).
        plan_no_filters = self._strip_filter_nodes(plan)
        if self._find_aggregate(plan_no_filters) is None and not isinstance(plan_no_filters, Project):
            result = self._plan_to_table(plan)
            if post_ops:
                result = self._apply_post_ops_custom(result, post_ops)
            t1 = _time.perf_counter()
            if verbose:
                print(f"[mxframe] PyArrow shortcut — {(t1-t0)*1000:.1f}ms")
            return result

        # 1 -- Check input-prep cache (fastest path) ------------------
        original_scan = self._find_scan(plan)
        plan_clean = self._strip_filter_nodes(plan)
        prep_key = (id(original_scan.table), original_scan.table.num_rows,
                    plan_clean.signature(), self._session_device)

        cached_prep = _INPUT_PREP_CACHE.get(prep_key)
        if cached_prep is not None:
            col_names = cached_prep['col_names']
            col_arrays = cached_prep['col_arrays']
            extra_inputs = cached_prep['extra_inputs']
            n_groups = cached_prep['n_groups']
            unique_key_cols = cached_prep['unique_key_cols']
            group_keys = cached_prep['group_keys']
            N = cached_prep['N']
            self._N = N
            self._maybe_switch_device(N)
        else:
            # 2 -- Pre-encode group keys on ORIGINAL table (cached) ----
            predicates = self._collect_predicates(plan)
            agg_node = self._find_aggregate(plan_clean)
            group_keys = []
            n_groups = 0
            unique_key_cols = {}
            group_ids_full = None

            if agg_node is not None and agg_node.group_by:
                group_keys = [e.args[0] for e in agg_node.group_by]
                group_ids_full, n_groups, unique_key_cols = self._build_group_ids_cached(
                    original_scan.table, group_keys
                )

            # 3 -- Compute filter mask (numpy boolean) -----------------
            mask_np = None
            if predicates:
                combined = None
                for pred in predicates:
                    m = self._eval_predicate(pred, original_scan.table)
                    combined = m if combined is None else pc.and_(combined, m)
                mask_np = combined.to_numpy(zero_copy_only=False)

            # 4 -- Apply mask + convert numeric columns to numpy -------
            N = int(mask_np.sum()) if mask_np is not None else original_scan.table.num_rows
            self._N = N
            self._maybe_switch_device(N)

            extra_inputs = {}
            if group_ids_full is not None:
                gids = group_ids_full[mask_np] if mask_np is not None else group_ids_full
                if mask_np is not None:
                    groups_present = np.zeros(n_groups, dtype=bool)
                    groups_present[gids] = True
                    if not groups_present.all():
                        present_idx = np.where(groups_present)[0]
                        remap = np.empty(n_groups, dtype=np.int32)
                        remap[present_idx] = np.arange(len(present_idx), dtype=np.int32)
                        gids = remap[gids]
                        n_groups = len(present_idx)
                        unique_key_cols = {
                            k: v.take(pa.array(present_idx.astype(np.int64)))
                            for k, v in unique_key_cols.items()
                        }
                extra_inputs["__group_ids__"] = gids

            col_names = []
            col_arrays = {}
            for name in original_scan.table.column_names:
                if not self._is_numeric_arrow_type(original_scan.table.schema.field(name).type):
                    continue
                np_arr, _, _ = self._column_to_numpy_cached(original_scan.table.column(name))
                if np_arr is None:
                    continue
                if mask_np is not None:
                    np_arr = np_arr[mask_np]
                col_names.append(name)
                col_arrays[name] = np_arr

            # Store in input-prep cache
            _INPUT_PREP_CACHE[prep_key] = {
                'col_names': col_names, 'col_arrays': col_arrays,
                'extra_inputs': extra_inputs, 'n_groups': n_groups,
                'unique_key_cols': unique_key_cols, 'group_keys': group_keys,
                'N': N,
            }

        all_names = col_names + list(extra_inputs.keys())
        all_arrays = {**col_arrays, **extra_inputs}

        # 5 -- Graph cache lookup --------------------------------------
        cache_key = self._compute_cache_key(plan_clean, all_names, all_arrays, n_groups)
        cache_hit = cache_key in _MODEL_CACHE

        if cache_hit:
            model, agg_names = _MODEL_CACHE[cache_key]
            t_compile = _time.perf_counter()
        else:
            if verbose:
                print("[mxframe] Compiling query plan...", end=" ", flush=True)

            input_types = [
                TensorType(_max_dtype(all_arrays[n]), list(all_arrays[n].shape), self._device_ref)
                for n in all_names
            ]

            graph = Graph(
                name="mxframe_custom",
                input_types=input_types,
                custom_extensions=[Path(self.kernels_path)],
            )
            with graph:
                nodes = {n: graph.inputs[i] for i, n in enumerate(all_names)}
                result_nodes = self._visit_plan_custom(plan_clean, nodes, n_groups=n_groups)
                graph.output(*result_nodes.values())

            model = self.session.load(graph)
            agg_names = list(result_nodes.keys())
            _MODEL_CACHE[cache_key] = (model, agg_names)
            t_compile = _time.perf_counter()

        # 6 -- Execute -------------------------------------------------
        gpu_input_path = "cpu_numpy"
        if self._session_device == "gpu":
            execute_inputs = []
            gpu_paths = []
            for n in all_names:
                gpu_in, path_used = self._to_gpu_input(all_arrays[n])
                execute_inputs.append(gpu_in)
                gpu_paths.append(path_used)
            if gpu_paths:
                gpu_input_path = gpu_paths[0] if len(set(gpu_paths)) == 1 else "mixed"
        else:
            execute_inputs = [all_arrays[n] for n in all_names]

        output_vals = model.execute(*execute_inputs)
        t_exec = _time.perf_counter()

        if verbose:
            compile_ms = (t_compile - t0) * 1000
            exec_ms = (t_exec - t_compile) * 1000
            total_ms = (t_exec - t0) * 1000
            if cache_hit:
                print(f"[mxframe] Cache hit. Executed on {self._session_device.upper()} in "
                      f"{total_ms:.1f}ms (data prep: {compile_ms:.1f}ms, execute: {exec_ms:.1f}ms)")
            else:
                print(f"done in {total_ms:.1f}ms "
                      f"(compile: {compile_ms:.1f}ms, execute: {exec_ms:.1f}ms). "
                      f"Next run with same schema: ~{exec_ms:.0f}ms")

        self.last_compile_provenance = {
            "device": self._session_device,
            "rows": int(N),
            "grouped": bool(group_keys),
            "n_groups": int(n_groups),
            "cache_hit": cache_hit,
            "compile_ms": round((t_compile - t0) * 1000, 1),
            "execute_ms": round((t_exec - t_compile) * 1000, 1),
            "gpu_input_path": gpu_input_path,
        }

        # 7 -- Assemble output -----------------------------------------
        agg_arrays = [pa.array(np.asarray(t.to_numpy()).reshape(-1)) for t in output_vals]

        if unique_key_cols:
            key_arrays = [unique_key_cols[k] for k in group_keys]
            result = pa.Table.from_arrays(key_arrays + agg_arrays, names=group_keys + agg_names)
        else:
            result = pa.Table.from_arrays(agg_arrays, names=agg_names)

        # 8 -- Apply post-ops (Sort/Limit/Distinct via Mojo kernels) ---
        if post_ops:
            result = self._apply_post_ops_custom(result, post_ops)

        return result

    def _visit_plan_custom(self, plan, nodes, *, n_groups=0):
        if isinstance(plan, Scan):
            return nodes
        elif isinstance(plan, Project):
            upstream = self._visit_plan_custom(plan.input, nodes, n_groups=n_groups)
            out = {}
            for i, expr in enumerate(plan.exprs):
                name = expr._alias or f'col_{i}'
                out[name] = self._visit_expr(expr, upstream)
            return out
        elif isinstance(plan, Filter):
            raise RuntimeError(
                'Filter node reached _visit_plan_custom -- '
                '_strip_filters should have removed all Filter nodes.'
            )
        elif isinstance(plan, (Sort, Limit, Distinct)):
            # Post-ops: should have been extracted by _extract_post_ops.
            # If they reach here, just recurse through to inner plan.
            return self._visit_plan_custom(plan.input, nodes, n_groups=n_groups)
        elif isinstance(plan, Aggregate):
            return self._visit_aggregate_custom(plan, nodes, n_groups=n_groups)
        raise NotImplementedError(f'Unsupported plan node: {type(plan)}')

    def _visit_aggregate_custom(self, plan, nodes, *, n_groups):
        upstream = self._visit_plan_custom(plan.input, nodes, n_groups=n_groups)
        out = {}
        grouped = bool(plan.group_by)
        on_gpu = (self._session_device == "gpu")

        for i, expr in enumerate(plan.aggs):
            name = expr._alias or f'agg_{i}'
            if grouped and n_groups > 0:
                gid_node = ops.cast(upstream["__group_ids__"], DType.int32)

                if on_gpu:
                    if n_groups <= 0:
                        raise ValueError(
                            f"Grouped GPU path requires n_groups > 0; got {n_groups}."
                        )
                    if n_groups > MAX_GROUPS:
                        raise ValueError(
                            f"Grouped GPU path supports up to {MAX_GROUPS} groups per launch; "
                            f"got {n_groups}. Split/chunk groups before GPU execution."
                        )
                    N = getattr(self, '_N', 0)
                    num_warps = ceil(N / WARP_SIZE)
                    if num_warps <= 0:
                        raise ValueError(
                            f"Grouped GPU path requires at least one warp; got N={N}, num_warps={num_warps}."
                        )

                    if expr.op == "count":
                        out_type = TensorType(DType.float32, [num_warps * n_groups], self._device_ref)
                        partial = ops.custom(
                            name="group_count",
                            values=[gid_node],
                            out_types=[out_type],
                            device=self._device_ref,
                        )[0]
                        out[name] = ops.sum(ops.reshape(partial, [num_warps, n_groups]), axis=0)

                    elif expr.op in ("sum", "min", "max"):
                        out_type = TensorType(DType.float32, [num_warps * n_groups], self._device_ref)
                        val_node = ops.cast(self._visit_expr(expr.args[0], upstream), DType.float32)
                        partial = ops.custom(
                            name=f"group_{expr.op}",
                            values=[val_node, gid_node],
                            out_types=[out_type],
                            device=self._device_ref,
                        )[0]
                        reshaped = ops.reshape(partial, [num_warps, n_groups])
                        reduce_fn = ops.sum if expr.op == "sum" else (ops.min if expr.op == "min" else ops.max)
                        out[name] = reduce_fn(reshaped, axis=0)

                    elif expr.op == "mean":
                        out_type = TensorType(DType.float32, [num_warps * n_groups], self._device_ref)
                        val_node = ops.cast(self._visit_expr(expr.args[0], upstream), DType.float32)
                        sum_partial = ops.custom(
                            name="group_sum",
                            values=[val_node, gid_node],
                            out_types=[out_type],
                            device=self._device_ref,
                        )[0]
                        sum_node = ops.sum(ops.reshape(sum_partial, [num_warps, n_groups]), axis=0)
                        cnt_partial = ops.custom(
                            name="group_count",
                            values=[gid_node],
                            out_types=[out_type],
                            device=self._device_ref,
                        )[0]
                        cnt_node = ops.sum(ops.reshape(cnt_partial, [num_warps, n_groups]), axis=0)
                        out[name] = ops.div(sum_node, ops.cast(cnt_node, DType.float32))

                    else:
                        raise NotImplementedError(f"Grouped '{expr.op}' GPU path not yet implemented.")

                else:
                    # CPU path (unchanged from Phase 1)
                    out_type = TensorType(DType.float32, [n_groups], self._device_ref)
                    if expr.op == "count":
                        out[name] = ops.custom(
                            name="group_count",
                            values=[gid_node],
                            out_types=[out_type],
                            device=self._device_ref,
                        )[0]
                    elif expr.op in ("sum", "min", "max", "mean"):
                        val_node = ops.cast(self._visit_expr(expr.args[0], upstream), DType.float32)
                        out[name] = ops.custom(
                            name=f"group_{expr.op}",
                            values=[val_node, gid_node],
                            out_types=[out_type],
                            device=self._device_ref,
                        )[0]
                    else:
                        raise NotImplementedError(
                            f"Grouped '{expr.op}' is not yet supported. "
                            f"Add a group_{expr.op}.mojo kernel and wire it."
                        )
            else:
                out[name] = self._visit_expr(expr, upstream)
        return out

    @staticmethod
    def _find_aggregate(plan):
        if isinstance(plan, Aggregate): return plan
        if hasattr(plan, 'input'): return CustomOpsCompiler._find_aggregate(plan.input)
        return None

    # -- Sort / Limit / Distinct via Mojo kernels ----------------------

    def _encode_sort_key(self, table: pa.Table, by_exprs, descending: list) -> np.ndarray:
        """Dictionary-encode sort key columns into a single int32 composite key.

        Each sort key column is dictionary-encoded to contiguous int32 indices,
        then remapped so that ascending index = lexicographic order.
        For multi-key sorts, indices are composed with strides so that the
        composite integer has the correct lexicographic ordering.
        For descending keys, the index is flipped so ascending sort -> descending.
        """
        key_names = [e.args[0] for e in by_exprs]
        encodings = []
        for k in key_names:
            arr = table.column(k)
            if isinstance(arr, pa.ChunkedArray):
                arr = arr.combine_chunks()
            encodings.append(arr.dictionary_encode())

        sizes = [len(e.dictionary) for e in encodings]

        # Compute strides for lexicographic composite key
        strides = []
        for i in range(len(sizes)):
            s = 1
            for j in range(i + 1, len(sizes)):
                s *= sizes[j]
            strides.append(s)

        composite = np.zeros(len(table), dtype=np.int32)
        for i, enc in enumerate(encodings):
            # Remap dictionary indices so ascending index = lexicographic order.
            # dictionary_encode() gives encounter-order, not sorted-order.
            dict_arr = enc.dictionary
            sorted_order = pc.sort_indices(dict_arr).to_numpy(zero_copy_only=False)
            # sorted_order[rank] = original_dict_idx  ->  we need rank_of[original_dict_idx] = rank
            rank_of = np.empty(len(dict_arr), dtype=np.int32)
            for r, orig_idx in enumerate(sorted_order):
                rank_of[orig_idx] = r
            raw_idx = enc.indices.to_numpy(zero_copy_only=False).astype(np.int32)
            idx = rank_of[raw_idx]
            if descending[i]:
                idx = (sizes[i] - 1) - idx  # flip so ascending sort -> descending
            composite += idx * strides[i]
        return composite

    def _apply_sort_custom(self, table: pa.Table, sort_node: Sort) -> pa.Table:
        """Sort an Arrow table by integer composite key.

        CPU: pure numpy argsort -- no MAX Graph overhead.
        GPU: Mojo bitonic sort kernel via MAX Graph, model cached by N.
        """
        composite_key = self._encode_sort_key(table, sort_node.by, sort_node.descending)
        N = len(composite_key)

        if self._session_device != "gpu":
            # CPU fast path: numpy argsort, no graph compilation
            sorted_indices = np.argsort(composite_key, kind='stable')
        else:
            # GPU path: Mojo bitonic sort, model cached by N
            cache_key = ("sort_indices", N, self._session_device)
            model = _POST_OP_MODEL_CACHE.get(cache_key)
            if model is None:
                key_type = TensorType(DType.int32, [N], self._device_ref)
                flag_type = TensorType(DType.int32, [1], self._device_ref)
                graph = Graph(
                    name="mxframe_sort",
                    input_types=[key_type, flag_type],
                    custom_extensions=[Path(self.kernels_path)],
                )
                with graph:
                    indices = ops.custom(
                        name="sort_indices",
                        values=[graph.inputs[0], graph.inputs[1]],
                        out_types=[TensorType(DType.int32, [N], self._device_ref)],
                        device=self._device_ref,
                    )[0]
                    graph.output(indices)
                model = self.session.load(graph)
                _POST_OP_MODEL_CACHE[cache_key] = model
            desc_flag = np.array([0], dtype=np.int32)
            k_buf = driver.Buffer.from_numpy(composite_key).to(self._gpu_driver)
            f_buf = driver.Buffer.from_numpy(desc_flag).to(self._gpu_driver)
            (out_idx,) = model.execute(k_buf, f_buf)
            sorted_indices = np.asarray(out_idx.to_numpy()).reshape(-1)

        return table.take(pa.array(sorted_indices.astype(np.int64)))

    def _apply_distinct_custom(self, table: pa.Table, distinct_node: Distinct) -> pa.Table:
        """Remove duplicate rows.

        CPU: numpy sort + consecutive-diff mask -- no MAX Graph overhead.
        GPU: Mojo sort_indices + unique_mask kernels, models cached by N.
        """
        cols = distinct_node.subset or table.column_names
        by_exprs = [Expr("col", c) for c in cols]
        desc = [False] * len(cols)
        composite_key = self._encode_sort_key(table, by_exprs, desc)
        N = len(composite_key)

        if self._session_device != "gpu":
            # CPU fast path: pure numpy, no graph compilation
            sort_order = np.argsort(composite_key, kind='stable')
            sorted_key = composite_key[sort_order]
            first = np.empty(N, dtype=bool)
            first[0] = True
            first[1:] = sorted_key[1:] != sorted_key[:-1]
            keep = sort_order[first]
            return table.take(pa.array(keep.astype(np.int64)))
        else:
            # GPU path: Mojo kernels, models cached by N
            sort_node_tmp = Sort(Scan(table), by_exprs, desc)
            sorted_table = self._apply_sort_custom(table, sort_node_tmp)
            sorted_key = self._encode_sort_key(sorted_table, by_exprs, desc)

            cache_key = ("unique_mask", N, self._session_device)
            model = _POST_OP_MODEL_CACHE.get(cache_key)
            if model is None:
                key_type = TensorType(DType.int32, [N], self._device_ref)
                graph = Graph(
                    name="mxframe_unique",
                    input_types=[key_type],
                    custom_extensions=[Path(self.kernels_path)],
                )
                with graph:
                    mask = ops.custom(
                        name="unique_mask",
                        values=[graph.inputs[0]],
                        out_types=[TensorType(DType.int32, [N], self._device_ref)],
                        device=self._device_ref,
                    )[0]
                    graph.output(mask)
                model = self.session.load(graph)
                _POST_OP_MODEL_CACHE[cache_key] = model

            k_buf = driver.Buffer.from_numpy(sorted_key).to(self._gpu_driver)
            (out_mask,) = model.execute(k_buf)
            mask_np = np.asarray(out_mask.to_numpy()).reshape(-1).astype(bool)
            return sorted_table.filter(pa.array(mask_np))

    def _apply_post_ops_custom(self, table: pa.Table, post_ops: list) -> pa.Table:
        """Apply Sort/Limit/Distinct using Mojo kernels (not PyArrow compute)."""
        for node in post_ops:
            if isinstance(node, Sort):
                table = self._apply_sort_custom(table, node)
            elif isinstance(node, Limit):
                table = table.slice(0, node.n)
            elif isinstance(node, Distinct):
                table = self._apply_distinct_custom(table, node)
        return table


    # -- Join materialization ------------------------------------------

    def _materialize_joins(self, plan):
        """Recursively replace Join nodes with Scan(joined_table).

        Walks bottom-up: materializes both sides of a Join, executes the
        hash join (Mojo kernel or numpy), and returns a Scan of the result.
        After this, the plan tree is a single-input chain again.
        """
        if isinstance(plan, Scan):
            return plan
        if isinstance(plan, Join):
            left_plan = self._materialize_joins(plan.left)
            right_plan = self._materialize_joins(plan.right)
            left_table = self._plan_to_table(left_plan)
            right_table = self._plan_to_table(right_plan)
            joined = self._execute_hash_join(
                left_table, right_table, plan.left_on, plan.right_on
            )
            return Scan(joined)
        if isinstance(plan, Filter):
            return Filter(self._materialize_joins(plan.input), plan.predicate)
        if isinstance(plan, Project):
            return Project(self._materialize_joins(plan.input), plan.exprs)
        if isinstance(plan, Aggregate):
            return Aggregate(self._materialize_joins(plan.input), plan.group_by, plan.aggs)
        if isinstance(plan, Sort):
            return Sort(self._materialize_joins(plan.input), plan.by, plan.descending)
        if isinstance(plan, Limit):
            return Limit(self._materialize_joins(plan.input), plan.n)
        if isinstance(plan, Distinct):
            return Distinct(self._materialize_joins(plan.input), plan.subset)
        if hasattr(plan, 'input'):
            plan.input = self._materialize_joins(plan.input)
        return plan

    def _plan_to_table(self, plan) -> pa.Table:
        """Resolve a join-free plan subtree to a concrete pa.Table.

        Handles:
        - Scan: return table directly
        - Filter → ... → Scan: apply predicates eagerly
        - Complex plans with Aggregate/Project: delegate to compile_and_run
        """
        if isinstance(plan, Scan):
            return plan.table
        # For Filter → Scan chains, apply predicates in PyArrow
        if isinstance(plan, Filter):
            inner_table = self._plan_to_table(plan.input)
            mask = self._eval_predicate(plan.predicate, inner_table)
            return inner_table.filter(mask)
        # For anything more complex, use the full compiler
        return self.compile_and_run(plan)

    def _execute_hash_join(self, left_table, right_table, left_on, right_on):
        """Execute an inner hash join using Mojo kernels (GPU) or numpy (CPU).

        Returns the joined pa.Table with all left columns + non-key right columns.
        """
        # Encode join keys as int32 arrays
        left_keys = self._encode_join_keys(left_table, left_on)
        right_keys = self._encode_join_keys(right_table, right_on)

        if self._session_device != "gpu":
            left_idx, right_idx = self._hash_join_mojo_cpu(left_keys, right_keys)
        else:
            left_idx, right_idx = self._hash_join_mojo_gpu(left_keys, right_keys)

        # Assemble joined table
        left_take = left_table.take(pa.array(left_idx.astype(np.int64)))
        right_take = right_table.take(pa.array(right_idx.astype(np.int64)))

        # Combine: all left columns + right columns excluding join keys
        arrays = []
        names = []
        for col_name in left_table.column_names:
            arrays.append(left_take.column(col_name))
            names.append(col_name)
        for col_name in right_table.column_names:
            if col_name not in right_on:
                arrays.append(right_take.column(col_name))
                names.append(col_name)

        return pa.Table.from_arrays(arrays, names=names)

    @staticmethod
    def _encode_join_keys(table, key_cols):
        """Encode join key columns into a single int32 numpy array.

        For single integer key: extract directly.
        For multi-key or non-integer: composite encode via dictionary encoding.
        """
        if len(key_cols) == 1:
            arr = table.column(key_cols[0])
            if isinstance(arr, pa.ChunkedArray):
                arr = arr.combine_chunks()
            # If already integer type, use directly
            if pa.types.is_integer(arr.type):
                return arr.to_numpy(zero_copy_only=False).astype(np.int32)
            # For non-integer, dictionary encode
            enc = arr.dictionary_encode()
            return enc.indices.to_numpy(zero_copy_only=False).astype(np.int32)

        # Multi-key: composite encoding (same approach as groupby)
        encodings = []
        for k in key_cols:
            arr = table.column(k)
            if isinstance(arr, pa.ChunkedArray):
                arr = arr.combine_chunks()
            encodings.append(arr.dictionary_encode())

        sizes = [len(e.dictionary) for e in encodings]
        strides = []
        for i in range(len(sizes)):
            s = 1
            for j in range(i + 1, len(sizes)):
                s *= sizes[j]
            strides.append(s)

        composite = np.zeros(len(table), dtype=np.int32)
        for i, enc in enumerate(encodings):
            idx = enc.indices.to_numpy(zero_copy_only=False)
            composite += (idx * strides[i]).astype(np.int32)
        return composite

    def _hash_join_mojo_cpu(self, left_keys, right_keys):
        """CPU path: two-pass hash join via Mojo kernels through MAX Graph.

        Uses join_count_cpu and join_scatter_cpu Mojo kernels — true Mojo-first,
        not numpy.  Returns (left_indices, right_indices) as int32 numpy arrays.
        """
        n_left = len(left_keys)
        n_right = len(right_keys)
        if n_left == 0 or n_right == 0:
            return np.array([], dtype=np.int32), np.array([], dtype=np.int32)

        left_keys_i32 = left_keys.astype(np.int32)
        right_keys_i32 = right_keys.astype(np.int32)

        # -- Pass 1: Count matches via Mojo CPU kernel --
        cache_key_count = ("join_count_cpu", n_left, n_right, self._session_device)
        model_count = _POST_OP_MODEL_CACHE.get(cache_key_count)
        if model_count is None:
            lk_type = TensorType(DType.int32, [n_left], self._device_ref)
            rk_type = TensorType(DType.int32, [n_right], self._device_ref)
            graph = Graph(
                name="mxframe_join_count_cpu",
                input_types=[lk_type, rk_type],
                custom_extensions=[Path(self.kernels_path)],
            )
            with graph:
                match_counts_node = ops.custom(
                    name="join_count_cpu",
                    values=[graph.inputs[0], graph.inputs[1]],
                    out_types=[
                        TensorType(DType.int32, [n_left], self._device_ref),
                    ],
                    device=self._device_ref,
                )[0]
                graph.output(match_counts_node)
            model_count = self.session.load(graph)
            _POST_OP_MODEL_CACHE[cache_key_count] = model_count

        (match_counts_out,) = model_count.execute(left_keys_i32, right_keys_i32)
        match_counts = np.asarray(match_counts_out.to_numpy()).reshape(-1).astype(np.int32)

        # -- Python bridge: total + offsets --
        total = int(match_counts.sum())
        if total == 0:
            return np.array([], dtype=np.int32), np.array([], dtype=np.int32)

        offsets = np.zeros(n_left, dtype=np.int32)
        np.cumsum(match_counts[:-1], out=offsets[1:])

        # -- Pass 2: Scatter via Mojo CPU kernel --
        cache_key_scatter = ("join_scatter_cpu", n_left, n_right, total, self._session_device)
        model_scatter = _POST_OP_MODEL_CACHE.get(cache_key_scatter)
        if model_scatter is None:
            lk_type = TensorType(DType.int32, [n_left], self._device_ref)
            rk_type = TensorType(DType.int32, [n_right], self._device_ref)
            off_type = TensorType(DType.int32, [n_left], self._device_ref)
            graph = Graph(
                name="mxframe_join_scatter_cpu",
                input_types=[lk_type, rk_type, off_type],
                custom_extensions=[Path(self.kernels_path)],
            )
            with graph:
                left_out_node, right_out_node = ops.custom(
                    name="join_scatter_cpu",
                    values=[graph.inputs[0], graph.inputs[1], graph.inputs[2]],
                    out_types=[
                        TensorType(DType.int32, [total], self._device_ref),
                        TensorType(DType.int32, [total], self._device_ref),
                    ],
                    device=self._device_ref,
                )
                graph.output(left_out_node, right_out_node)
            model_scatter = self.session.load(graph)
            _POST_OP_MODEL_CACHE[cache_key_scatter] = model_scatter

        left_out_t, right_out_t = model_scatter.execute(left_keys_i32, right_keys_i32, offsets)
        left_idx = np.asarray(left_out_t.to_numpy()).reshape(-1).astype(np.int32)
        right_idx = np.asarray(right_out_t.to_numpy()).reshape(-1).astype(np.int32)

        return left_idx, right_idx

    def _hash_join_mojo_gpu(self, left_keys, right_keys):
        """GPU path: two-pass hash join via Mojo kernels through MAX Graph.

        Pass 1 (join_count): count matches per left row.
        Python bridge: sum → total, prefix sum → offsets.
        Pass 2 (join_scatter): emit index pairs using pre-built position arrays.
        """
        n_left = len(left_keys)
        n_right = len(right_keys)
        max_key = max(int(left_keys.max()), int(right_keys.max())) if n_left > 0 and n_right > 0 else 0
        table_size = max_key + 1

        left_keys_i32 = left_keys.astype(np.int32)
        right_keys_i32 = right_keys.astype(np.int32)

        # -- Pass 1: Count matches via GPU kernel --
        max_key_buf = np.array([max_key], dtype=np.int32)

        cache_key_count = ("join_count_gpu", n_left, n_right, table_size, self._session_device)
        model_count = _POST_OP_MODEL_CACHE.get(cache_key_count)
        if model_count is None:
            lk_type = TensorType(DType.int32, [n_left], self._device_ref)
            rk_type = TensorType(DType.int32, [n_right], self._device_ref)
            mk_type = TensorType(DType.int32, [1], self._device_ref)
            graph = Graph(
                name="mxframe_join_count",
                input_types=[lk_type, rk_type, mk_type],
                custom_extensions=[Path(self.kernels_path)],
            )
            with graph:
                match_counts_node, right_count_node = ops.custom(
                    name="join_count_gpu",
                    values=[graph.inputs[0], graph.inputs[1], graph.inputs[2]],
                    out_types=[
                        TensorType(DType.int32, [n_left], self._device_ref),
                        TensorType(DType.int32, [table_size], self._device_ref),
                    ],
                    device=self._device_ref,
                )
                graph.output(match_counts_node, right_count_node)
            model_count = self.session.load(graph)
            _POST_OP_MODEL_CACHE[cache_key_count] = model_count

        lk_buf = driver.Buffer.from_numpy(left_keys_i32).to(self._gpu_driver)
        rk_buf = driver.Buffer.from_numpy(right_keys_i32).to(self._gpu_driver)
        mk_buf = driver.Buffer.from_numpy(max_key_buf).to(self._gpu_driver)
        match_counts_out, _ = model_count.execute(lk_buf, rk_buf, mk_buf)
        match_counts = np.asarray(match_counts_out.to_numpy()).reshape(-1).astype(np.int32)

        # -- Python bridge --
        total = int(match_counts.sum())
        if total == 0:
            return np.array([], dtype=np.int32), np.array([], dtype=np.int32)

        offsets = np.zeros(n_left, dtype=np.int32)
        np.cumsum(match_counts[:-1], out=offsets[1:])

        # -- Build right-side position arrays for GPU scatter --
        right_count = np.bincount(right_keys_i32, minlength=table_size).astype(np.int32)
        right_order = np.argsort(right_keys_i32, kind='stable').astype(np.int32)
        right_starts = np.zeros(table_size, dtype=np.int32)
        np.cumsum(right_count[:-1], out=right_starts[1:])

        # -- Pass 2: Scatter via GPU kernel --
        cache_key_scatter = ("join_scatter_gpu", n_left, n_right, total, table_size, self._session_device)
        model_scatter = _POST_OP_MODEL_CACHE.get(cache_key_scatter)
        if model_scatter is None:
            lk_type = TensorType(DType.int32, [n_left], self._device_ref)
            rk_type = TensorType(DType.int32, [n_right], self._device_ref)
            off_type = TensorType(DType.int32, [n_left], self._device_ref)
            rsi_type = TensorType(DType.int32, [n_right], self._device_ref)
            rks_type = TensorType(DType.int32, [table_size], self._device_ref)
            rkc_type = TensorType(DType.int32, [table_size], self._device_ref)
            graph = Graph(
                name="mxframe_join_scatter",
                input_types=[lk_type, rk_type, off_type, rsi_type, rks_type, rkc_type],
                custom_extensions=[Path(self.kernels_path)],
            )
            with graph:
                left_out_node, right_out_node = ops.custom(
                    name="join_scatter_gpu",
                    values=[graph.inputs[0], graph.inputs[1], graph.inputs[2],
                            graph.inputs[3], graph.inputs[4], graph.inputs[5]],
                    out_types=[
                        TensorType(DType.int32, [total], self._device_ref),
                        TensorType(DType.int32, [total], self._device_ref),
                    ],
                    device=self._device_ref,
                )
                graph.output(left_out_node, right_out_node)
            model_scatter = self.session.load(graph)
            _POST_OP_MODEL_CACHE[cache_key_scatter] = model_scatter

        off_buf = driver.Buffer.from_numpy(offsets).to(self._gpu_driver)
        rsi_buf = driver.Buffer.from_numpy(right_order).to(self._gpu_driver)
        rks_buf = driver.Buffer.from_numpy(right_starts).to(self._gpu_driver)
        rkc_buf = driver.Buffer.from_numpy(right_count).to(self._gpu_driver)
        left_out_t, right_out_t = model_scatter.execute(lk_buf, rk_buf, off_buf, rsi_buf, rks_buf, rkc_buf)
        left_idx = np.asarray(left_out_t.to_numpy()).reshape(-1).astype(np.int32)
        right_idx = np.asarray(right_out_t.to_numpy()).reshape(-1).astype(np.int32)

        return left_idx, right_idx

    @staticmethod
    def _build_group_ids(table, keys):
        """Dictionary-encode group keys into contiguous int32 group ids."""
        if len(keys) == 1:
            col_arr = table.column(keys[0])
            if isinstance(col_arr, pa.ChunkedArray):
                col_arr = col_arr.combine_chunks()
            encoded = col_arr.dictionary_encode()
            ids = encoded.indices.to_numpy(zero_copy_only=False).astype(np.int32, copy=False)
            return ids, len(encoded.dictionary), {keys[0]: encoded.dictionary}

        encodings = []
        for k in keys:
            arr = table.column(k)
            if isinstance(arr, pa.ChunkedArray):
                arr = arr.combine_chunks()
            encodings.append(arr.dictionary_encode())

        sizes = [len(e.dictionary) for e in encodings]
        strides = []
        for i in range(len(sizes)):
            s = 1
            for j in range(i + 1, len(sizes)):
                s *= sizes[j]
            strides.append(s)
        max_groups = strides[0] * sizes[0]

        composite = np.zeros(len(table), dtype=np.int32)
        for i, enc in enumerate(encodings):
            idx = enc.indices.to_numpy(zero_copy_only=False)
            composite += (idx * strides[i]).astype(np.int32)

        if max_groups <= 1024:
            present_mask = np.zeros(max_groups, dtype=bool)
            present_mask[composite] = True
            present = np.where(present_mask)[0]
            n_groups = len(present)

            if n_groups == max_groups:
                dense_ids = composite
            else:
                remap = np.empty(max_groups, dtype=np.int32)
                remap[present] = np.arange(n_groups, dtype=np.int32)
                dense_ids = remap[composite]

            unique_key_cols = {}
            for i, k in enumerate(keys):
                key_idx = (present // strides[i]) % sizes[i]
                unique_key_cols[k] = encodings[i].dictionary.take(
                    pa.array(key_idx.astype(np.int64))
                )
            return dense_ids, n_groups, unique_key_cols

        composite_i64 = composite.astype(np.int64)
        unique_composite, dense_ids = np.unique(composite_i64, return_inverse=True)
        dense_ids = dense_ids.astype(np.int32)
        n_groups = len(unique_composite)

        unique_key_cols = {}
        for i, k in enumerate(keys):
            key_idx = ((unique_composite // strides[i]) % sizes[i]).astype(np.int64)
            unique_key_cols[k] = encodings[i].dictionary.take(pa.array(key_idx))

        return dense_ids, n_groups, unique_key_cols

    @staticmethod
    def _build_group_ids_cached(table, keys):
        """Cached wrapper: reuses group encoding when the same table object is queried."""
        cache_key = (id(table), table.num_rows, tuple(keys))
        cached = _GROUP_ENCODE_CACHE.get(cache_key)
        if cached is not None:
            return cached
        result = CustomOpsCompiler._build_group_ids(table, keys)
        _GROUP_ENCODE_CACHE[cache_key] = result
        return result


## Tests 🧪

Let's verify that our compiler can translate a simple plan and execute it.

In [ ]:
import pyarrow as pa
from mxframe.lazy_expr import col, lit
from mxframe.lazy_frame import LazyFrame, Scan

kernels_path = (
    str(Path(__file__).parent.parent / "mxframe" / "kernels_v261")
    if "__file__" in dir()
    else str(Path("/home/ablearn/mxdf/mxframe/kernels_v261"))
)

compiler = CustomOpsCompiler(kernels_path)

# ── Test 1: Projection falls through to built-in ops ──
table = pa.table({'a': [1, 2, 3], 'b': [4, 5, 6]})
lf = LazyFrame(Scan(table))
result = compiler.compile_and_run(lf.select(col('a') + lit(10)).plan)
assert result.num_columns == 1
assert result.column(0).to_pylist() == [11, 12, 13], f'Projection failed: {result}'
print('✅ Test 1 passed: projection')

# ── Test 2: Global sum via built-in ops ──
table2 = pa.table({'x': [1.0, 2.0, 3.0, 4.0]})
lf2 = LazyFrame(Scan(table2))
result2 = compiler.compile_and_run(
    lf2.groupby().agg(col('x').sum().alias('total')).plan
)
assert abs(result2.column('total').to_pylist()[0] - 10.0) < 1e-6, f'Sum failed: {result2}'
print('✅ Test 2 passed: global sum aggregation')

# ── Test 3: Grouped sum via Mojo group_sum kernel -- with group keys in result ──
table3 = pa.table({
    'group': ['a', 'b', 'a', 'b', 'a'],
    'val':   [1.0, 2.0, 3.0, 4.0, 5.0],
})
lf3 = LazyFrame(Scan(table3))
result3 = compiler.compile_and_run(
    lf3.groupby('group').agg(col('val').sum().alias('total')).plan
)
assert 'group' in result3.column_names, f'Missing group key: {result3.column_names}'
assert result3.column_names[0] == 'group', f'Key should be first: {result3.column_names}'
groups = result3.column('group').to_pylist()
totals = result3.column('total').to_pylist()
result_dict = dict(zip(groups, totals))
assert abs(result_dict['a'] - 9.0) < 1e-6, f'Sum for a should be 9.0: {result_dict}'
assert abs(result_dict['b'] - 6.0) < 1e-6, f'Sum for b should be 6.0: {result_dict}'
print('✅ Test 3 passed: grouped sum with key columns in result (a=9, b=6)')

# ── Test 4: Filter removes rows before grouped aggregation ──
table4 = pa.table({
    'group': ['a', 'b', 'a', 'b', 'a'],
    'val':   [1.0, 2.0, 3.0, 4.0, 5.0],
    'flag':  [1,   1,   0,   1,   1  ],
})
lf4 = LazyFrame(Scan(table4))
result4 = compiler.compile_and_run(
    lf4.filter(col('flag') > lit(0)).groupby('group').agg(col('val').sum().alias('total')).plan
)
groups4 = result4.column('group').to_pylist()
totals4 = result4.column('total').to_pylist()
result_dict4 = dict(zip(groups4, totals4))
assert abs(result_dict4['a'] - 6.0) < 1e-6, f'Filtered sum for a should be 6.0: {result_dict4}'
assert abs(result_dict4['b'] - 6.0) < 1e-6, f'Filtered sum for b should be 6.0: {result_dict4}'
print('✅ Test 4 passed: filter removes rows before grouped aggregation')

# ─── Phase 1 tests ────────────────────────────────────────────────────────────
# Data: group x = [1.0, 2.0], group y = [3.0]
# min(x)=1  max(x)=2  mean(x)=1.5  count(x)=2
# min(y)=3  max(y)=3  mean(y)=3.0  count(y)=1
table5 = pa.table({'g': ['x', 'x', 'y'], 'v': [1.0, 2.0, 3.0]})
lf5 = LazyFrame(Scan(table5))

def _as_dict(result, key_col, val_col):
    return dict(zip(result.column(key_col).to_pylist(), result.column(val_col).to_pylist()))

# ── Test 5: Grouped min ──
r5 = compiler.compile_and_run(lf5.groupby('g').agg(col('v').min().alias('mn')).plan)
d5 = _as_dict(r5, 'g', 'mn')
assert abs(d5['x'] - 1.0) < 1e-5, f'min x should be 1.0: {d5}'
assert abs(d5['y'] - 3.0) < 1e-5, f'min y should be 3.0: {d5}'
print('✅ Test 5 passed: grouped min (x=1.0, y=3.0)')

# ── Test 6: Grouped max ──
r6 = compiler.compile_and_run(lf5.groupby('g').agg(col('v').max().alias('mx')).plan)
d6 = _as_dict(r6, 'g', 'mx')
assert abs(d6['x'] - 2.0) < 1e-5, f'max x should be 2.0: {d6}'
assert abs(d6['y'] - 3.0) < 1e-5, f'max y should be 3.0: {d6}'
print('✅ Test 6 passed: grouped max (x=2.0, y=3.0)')

# ── Test 7: Grouped mean ──
r7 = compiler.compile_and_run(lf5.groupby('g').agg(col('v').mean().alias('avg')).plan)
d7 = _as_dict(r7, 'g', 'avg')
assert abs(d7['x'] - 1.5) < 1e-5, f'mean x should be 1.5: {d7}'
assert abs(d7['y'] - 3.0) < 1e-5, f'mean y should be 3.0: {d7}'
print('✅ Test 7 passed: grouped mean (x=1.5, y=3.0)')

# ── Test 8: Grouped count ──
r8 = compiler.compile_and_run(lf5.groupby('g').agg(col('v').count().alias('cnt')).plan)
d8 = _as_dict(r8, 'g', 'cnt')
assert abs(d8['x'] - 2.0) < 1e-5, f'count x should be 2: {d8}'
assert abs(d8['y'] - 1.0) < 1e-5, f'count y should be 1: {d8}'
print('✅ Test 8 passed: grouped count (x=2, y=1)')

# ── Test 9: Multi-agg groupby (sum + mean in one call) ──
r9 = compiler.compile_and_run(
    lf5.groupby('g').agg(
        col('v').sum().alias('total'),
        col('v').mean().alias('avg'),
    ).plan
)
assert 'g' in r9.column_names and 'total' in r9.column_names and 'avg' in r9.column_names
d9_sum  = _as_dict(r9, 'g', 'total')
d9_mean = _as_dict(r9, 'g', 'avg')
assert abs(d9_sum['x']  - 3.0) < 1e-5, f'sum x should be 3.0: {d9_sum}'
assert abs(d9_mean['x'] - 1.5) < 1e-5, f'mean x should be 1.5: {d9_mean}'
print('✅ Test 9 passed: multi-agg groupby (sum + mean)')

print('\nAll CustomOpsCompiler tests passed! 🎉')


✅ Test 1 passed: projection
✅ Test 2 passed: global sum aggregation
✅ Test 3 passed: grouped sum with key columns in result (a=9, b=6)
✅ Test 4 passed: filter removes rows before grouped aggregation
✅ Test 5 passed: grouped min (x=1.0, y=3.0)
✅ Test 6 passed: grouped max (x=2.0, y=3.0)
✅ Test 7 passed: grouped mean (x=1.5, y=3.0)
✅ Test 8 passed: grouped count (x=2, y=1)
✅ Test 9 passed: multi-agg groupby (sum + mean)

All CustomOpsCompiler tests passed! 🎉
